In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *

In [2]:
data = pd.read_csv("../../resources/datasets/split_datasets/CLIP_X_test.csv", index_col=0)
data1 = torch.tensor(data.iloc[0:1].values, dtype=torch.float32)

rider_data = pd.read_csv("../../resources/datasets/split_datasets/aero_X_test.csv", index_col=0)
rider_data = rider_data[['upper_leg', 'lower_leg', 'arm_length', 'torso_length', 'neck_and_head_length', 'torso_width']]
rider1 = torch.tensor(rider_data.iloc[0:1].values, dtype=torch.float32)

# rider_full = 

In [3]:
StandardEvaluations: List[EvaluationFunction] = [
    UsabilityEvaluator(),
    AeroEvaluator(),
    ErgonomicsEvaluator(),
    AestheticsEvaluator(mode="Text"),
    StructuralEvaluator(),
    ValidationEvaluator(),
    FrameValidityEvaluator()
]



evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
condition = {"Rider": rider1, "Use Case": "mtb", "Text": "Sporty Bike"}

In [5]:
scores = evaluator(torch.tensor(data.values, dtype=torch.float32), condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [6]:
print(scores)

tensor([[   0.5265,   22.6199,    0.8513,  ...,  -76.6354,  -97.5000,
           -0.5000],
        [   0.2212,    7.7730,   62.0934,  ...,   76.3646, -179.0000,
           -0.5000],
        [   0.4809,   24.8670,    1.8961,  ..., -122.7396, -160.5000,
           -0.5000],
        ...,
        [   0.3368,   22.3032,    4.3629,  ...,  -52.7562, -176.0000,
           -0.5000],
        [   0.3506,   22.3979,    4.3632,  ...,  -58.1107, -176.0000,
           -0.5000],
        [   0.5507,   22.6195,    0.6076,  ...,  -81.2910, -100.0000,
           -0.5000]], grad_fn=<CopySlices>)


In [7]:
objective_scores = scores[:, isobjective]
constraint_scores = scores[:, ~isobjective]
print(objective_scores)
print(constraint_scores)

tensor([[ 0.5265, 22.6199,  0.8513,  ...,  2.0226,  1.2949,  1.3140],
        [ 0.2212,  7.7730, 62.0934,  ...,  0.9821,  2.1526,  2.1006],
        [ 0.4809, 24.8670,  1.8961,  ...,  0.8124,  2.3671,  2.5386],
        ...,
        [ 0.3368, 22.3032,  4.3629,  ...,  0.3775,  0.8025,  1.3432],
        [ 0.3506, 22.3979,  4.3632,  ...,  0.3321,  0.6779,  1.0987],
        [ 0.5507, 22.6195,  0.6076,  ...,  4.5765,  2.3677,  2.6635]],
       grad_fn=<IndexBackward0>)
tensor([[   1.2220,    0.7759, -139.3000,  ...,  -76.6354,  -97.5000,
           -0.5000],
        [   0.9097,    0.9507,  -25.0000,  ...,   76.3646, -179.0000,
           -0.5000],
        [   0.7730,    1.1335, -161.0000,  ..., -122.7396, -160.5000,
           -0.5000],
        ...,
        [   0.8024,    0.8337, -150.0000,  ...,  -52.7562, -176.0000,
           -0.5000],
        [   0.7746,    0.8257, -150.0000,  ...,  -58.1107, -176.0000,
           -0.5000],
        [   1.3636,    1.0577, -150.7000,  ...,  -81.2910, -100.0

In [8]:
df_evaluator = construct_dataframe_evaluator(StandardEvaluations)

In [9]:
scores, requirement_types = df_evaluator(data, condition)
isobjective = np.array(isobjective).astype(np.bool_)

#index columns using boolean isobjective [True, False, ...]
objective_scores = scores.iloc[:, isobjective]
constraint_scores = scores.iloc[:, ~isobjective]

display(objective_scores)
display(constraint_scores)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


,Usability Score - 0 to 1,Drag Force,Knee Angle Error,Hip Angle Error,Arm Angle Error,Cosine Similarity to Text,Mass,Planar Compliance,Transverse Compliance,Eccentric Compliance
1,0.526528,22.619877,0.851331,112.870125,162.000000,0.363277,3.046412,2.022574,1.294906,1.314018
2,0.221181,7.773015,62.093441,1.833075,126.239494,0.360625,0.494630,0.982076,2.152593,2.100595
3,0.480893,24.866962,1.896087,86.116112,162.000000,0.359375,2.986526,0.812391,2.367146,2.538648
4,0.646960,23.422394,21.904678,66.675972,23.378979,0.356008,1.755835,1.215829,2.208945,1.765494
5,0.226224,8.676568,59.083424,1.590315,19.168608,0.365683,2.734004,0.781487,1.306897,1.274086
...,...,...,...,...,...,...,...,...,...,...
4796,0.522583,25.245617,7.876009,62.071308,44.155533,0.367138,2.739317,2.320882,1.307140,1.480062
4797,0.654805,24.297882,8.097003,47.523190,1.721424,0.361898,3.115477,1.440918,2.149850,2.095844
4798,0.336759,22.303207,4.362941,129.039978,162.000000,0.359774,3.095254,0.377494,0.802486,1.343156
4799,0.350591,22.397942,4.363167,127.063225,162.000000,0.357441,3.472321,0.332086,0.677881,1.098729


,Planar Safety Factor,Eccentric Safety Factor,Saddle height too small,Seat post too short,Bsd rear too small,Bsd front too small,Head tube lower extension too great,Head tube length too great,Chain stay less than zero,Chain stay should be greater than wheel radius,Seat stay should be greater than wheel radius,The pedal shouldn't intersect the front wheel,The crank shouldn't hit the ground when it is in its lower position,Predicted Frame Validity
1,1.221981,0.775876,-139.299988,-150.700012,-40.0,-40.0,-85.600006,-45.600006,-430.000000,-119.000000,-184.669006,-76.635437,-97.500000,-0.500000
2,0.909706,0.950662,-25.000000,-65.000000,-40.0,-40.0,-56.900002,-33.599998,-350.000000,-96.500000,-20.948822,76.364594,-179.000000,-0.500000
3,0.773000,1.133458,-161.000000,-129.000000,-40.0,-40.0,-162.899994,-126.400002,-415.000000,-135.500000,-226.262878,-122.739563,-160.500000,-0.500000
4,0.949714,0.903975,-100.000000,-30.000000,-40.0,-40.0,-57.900002,-28.800003,-375.000000,-89.500000,-96.691803,82.112885,-114.500000,-0.500000
5,0.901327,0.811207,-10.000000,-230.000000,-40.0,-40.0,-28.799999,-8.599998,-431.790009,-152.290009,-80.747437,3.628601,-140.500000,-0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,1.283503,0.871472,-150.000000,-160.000000,-40.0,-40.0,-96.400002,-71.699997,-410.000000,-99.000000,-86.553772,-68.229187,-89.500000,-0.500000
4797,0.868196,0.818807,-300.000000,-155.000000,-22.0,-22.0,-51.000000,-27.000000,-419.000000,-127.000000,-92.111938,-158.037842,-121.590652,-0.500000
4798,0.802355,0.833713,-150.000000,-170.000000,-40.0,-40.0,-210.399994,-153.199997,-410.000000,-207.000000,-228.856445,-52.756165,-176.000000,-0.500000
4799,0.774629,0.825713,-150.000000,-170.000000,-40.0,-40.0,-269.600006,-222.399994,-410.000000,-207.000000,-228.859802,-58.110657,-176.000000,-0.500000
